# Capstone: Gad comparison

The comparison article, Gad et al. (2025, *Sensors* 25(19):6082), both ways:

1. **Replication** on their data: 21 CASAS homes, their features, LightGBM / XGBoost / CatBoost with
   their Table 3 settings, their random split; then the same with a time split, then with only what a
   home knows (`scripts/eval_gad.py`).
2. **Our method on their data**: our habit models on CASAS ("will this activity start in this hour?").
3. **Their method on our data**, and **the split effect**: baseline, random forest and their LightGBM on
   REFIT, UK-DALE and CASAS, split by time and at random (`scripts/eval_split_effect.py`).

**Settings:** Accelerator = **None** (CPU is enough; no GPU queue), Internet = **On** (downloads CASAS,
236 MB, from Zenodo), dataset `capstone-bundle` (latest version) under Input.
Save Version > Save & Run All (Commit). Phone notifications as before (secret `NTFY_URL`).

**A new notebook copies none of these settings.**

In [ ]:
import glob, os, shutil, subprocess
# os.environ['NTFY_URL'] = 'https://ntfy.sh/<your-topic>'   # used only if the NTFY_URL secret is missing
found = glob.glob('/kaggle/input/**/habits', recursive=True)
assert found, 'No bundle: add the capstone-bundle dataset with Add Input (right-hand panel), then rerun'
bundle = os.path.dirname(found[0])
shutil.copytree(bundle, '/kaggle/working/capstone', dirs_exist_ok=True)
os.chdir('/kaggle/working/capstone')
os.makedirs('logs', exist_ok=True)
print('bundle from', bundle)
!pip install -q paho-mqtt
import torch, sklearn, pandas, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pandas.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (check the Accelerator setting)')
import lightgbm, xgboost, catboost
print('lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| catboost', catboost.__version__)
sys.path.insert(0, 'scripts')
from kaggle_steps import ADDRESS, notify, run_step
print('ntfy:', 'on' if ADDRESS else 'off (no NTFY_URL secret or first-line topic)')
notify('Gad comparison: started', 'CASAS download, then the runs', 'default', 'rocket')

In [ ]:
!mkdir -p /kaggle/temp && wget -q -c -O /kaggle/temp/casas.zip "https://zenodo.org/records/15708568/files/labeled_data.zip?download=1"
!ls -l /kaggle/temp/casas.zip   # should be 236037656 bytes
ok = run_step('CASAS prepare', 'CAPSTONE_DATASET=casas python -u scripts/prepare_casas.py /kaggle/temp/casas.zip',
              'logs/casas_prepare.log', summary_lines=3)
casas_ok = ok  # without CASAS, its steps are skipped (REFIT and UK-DALE still run)

In [ ]:
DATASETS = ['refit', 'ukdale', 'casas']   # to rerun only CASAS: ['casas']
ok &= run_step('Tests', 'python -m pytest -q tests/test_gad.py', 'logs/tests.log', summary_lines=2)
if 'casas' in DATASETS and casas_ok:
    ok &= run_step('Gad replication (CASAS)', 'python -u scripts/eval_gad.py', 'logs/casas_gad.log',
                   env={'CAPSTONE_DATASET': 'casas'}, summary_lines=40)
    ok &= run_step('Our habit models on CASAS', 'python -u scripts/eval_refit.py', 'logs/casas_habit_models.log',
                   env={'CAPSTONE_DATASET': 'casas'}, summary_lines=30)
for ds in DATASETS:
    if ds != 'casas' or casas_ok:
        ok &= run_step(f'{ds} split effect', 'python -u scripts/eval_split_effect.py',
                       f'logs/{ds}_split_effect.log', env={'CAPSTONE_DATASET': ds}, summary_lines=20)

In [ ]:
import zipfile, pathlib
with zipfile.ZipFile('/kaggle/working/gad_comparison.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in [*pathlib.Path('data/results').rglob('*'), *pathlib.Path('logs').rglob('*')]:
        if f.is_file() and (f.suffix == '.log' or f.name.startswith(('split_effect', 'gad_')) or 'casas' in f.parts):
            zf.write(f, str(f))
print(round(pathlib.Path('/kaggle/working/gad_comparison.zip').stat().st_size / 1e6, 2), 'MB')
shutil.rmtree('/kaggle/working/capstone', ignore_errors=True)  # keep the Output to the zip
notify('Gad comparison: FINISHED' if ok else 'Gad comparison: finished WITH ERRORS',
       'gad_comparison.zip is in the Output tab', 'default' if ok else 'high', 'tada' if ok else 'warning')